In [4]:
# --- TUNABLE PARAMETERS ------------------------------------------------------
L = 4             # lattice linear size (4 instead of 8)
betas = [5.8, 6.0, 6.2, 6.4, 6.6]
epsilon = 0.02

burn_in    = 500     # was 2000
run_length = 10000   # was 40000
n_chains   = 3       # was 4
master_seed = 2025
# -----------------------------------------------------------------------------


def run_high_beta_test():
    print("Initializing T13 'High-Beta' test (multi-chain, light settings)...")

    # Cold start
    U0 = jnp.broadcast_to(
        jnp.eye(3, dtype=jnp.complex64),
        (L, L, L, L, 4, 3, 3),
    )

    master_key = jax.random.PRNGKey(master_seed)

    print("\n" + "=" * 90)
    print(f"{'Beta':<8} | {'<W> mean±std':<24} | {'tau mean±std':<24} | {'gap (1/tau)':<18}")
    print("=" * 90)

    beta_list = []
    tau_means = []
    tau_stds  = []

    for i, beta in enumerate(betas):
        beta_key = jax.random.fold_in(master_key, i)

        chain_means = []
        chain_taus  = []

        for c in range(n_chains):
            chain_key = jax.random.fold_in(beta_key, c)
            U = U0
            key = chain_key

            # Burn-in
            for _ in range(burn_in):
                key, sub = jax.random.split(key)
                U = langevin_step(U, sub, epsilon, beta)

            # Measurement trajectory
            w_hist = []
            for _ in range(run_length):
                key, sub = jax.random.split(key)
                U = langevin_step(U, sub, epsilon, beta)
                w = wilson_loop_trace(U, (0, 0, 0, 0), 2, 2)
                w_hist.append(float(w))

            w_hist = np.asarray(w_hist)
            tau = compute_integrated_autocorr(w_hist)
            chain_means.append(np.mean(w_hist))
            chain_taus.append(tau)

        chain_means = np.asarray(chain_means)
        chain_taus  = np.asarray(chain_taus)

        beta_list.append(beta)
        tau_means.append(chain_taus.mean())
        tau_stds.append(chain_taus.std(ddof=1) if n_chains > 1 else 0.0)

        gap_mean = 1.0 / chain_taus.mean() if chain_taus.mean() > 0 else 0.0

        print(
            f"{beta:<8.1f} | "
            f"{chain_means.mean():.4f} ± {chain_means.std(ddof=1):.4f} | "
            f"{chain_taus.mean():.4f} ± {chain_taus.std(ddof=1):.4f} | "
            f"{gap_mean:.4f}"
        )

    print("=" * 90)

    # ---------- FITTING WITH ERROR BARS (same as before) ----------
    tau_means_arr = np.array(tau_means)
    tau_stds_arr  = np.array(tau_stds)
    tau_stds_arr[tau_stds_arr == 0] = tau_means_arr.mean() * 0.05

    fit = fit_models(beta_list, tau_means_arr, tau_stds_arr)
    poly, exp = fit["poly"], fit["exp"]

    print("\n>>> MODEL COMPARISON (weighted fits) <<<")
    print(f"Polynomial: chi2 = {poly['chi2']:.3f}, red = {poly['red']:.3f}, AIC = {poly['AIC']:.3f}")
    print(f"Exponential: chi2 = {exp['chi2']:.3f}, red = {exp['red']:.3f}, AIC = {exp['AIC']:.3f}")

    if exp["AIC"] + 2 < poly["AIC"]:
        print("\nWinner: EXPONENTIAL (tunneling-like scaling in this window).")
    elif poly["AIC"] + 2 < exp["AIC"]:
        print("\nWinner: POLYNOMIAL (convexity-like scaling in this window).")
    else:
        print("\nResult: INCONCLUSIVE in this β-range with current statistics.")
